In [30]:
%load_ext autoreload
%autoreload 2
import parse_agent,pde_descriptions,system_prompt,coding_agent
import os
from dotenv import load_dotenv
from google import genai
from google.genai.types import GenerateContentConfig
import markdown_to_json
import json
from ollama import chat

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")


fk_description = pde_descriptions.fk_description
tau_d = 0.5714
pde_description = fk_description.format(tau_d=tau_d)

parse_prompt = parse_agent.parse_prompt
system_prompt = system_prompt.system_prompt

user_prompt = parse_prompt.format(
    user_input=pde_description,
)

response = chat(
    model='qwen2.5-coder:3b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)

#deleat ```json at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```json"):
    response_text = response_text[len("```json"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("parsed_resp.json", "w", encoding="utf-8") as f:
    f.write(response_text)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
```json
{
  "PDEs": "\\frac{\\partial u}{\\partial t} = D \\nabla^2 u - \\frac{I_{fi}(u, v) + I_{so}(u) + I_{si}(u, w)}{C_m}",
  "number_of_state_variables": 3,
  "texture_size": 512,
  "spatial_step": 0.0390625,
  "domain_size": 20.0,
  "temporal_step": 0.025,
  "time_horizon": 100.0,
  "boundary_conditions": "No-Flux",
  "parameter_values": {
    "D": 0.001,
    "C_m": 1.0,
    "tau_pv": 7.99,
    "tau_v1": 9.8,
    "tau_v2": 312.5,
    "tau_pw": 870.0,
    "tau_mw": 41.0,
    "tau_0": 12.5,
    "tau_r": 33.83,
    "tau_si": 29.0,
    "k": 10.0,
    "V_csi": 0.861,
    "V_v": 0.04
  },
  "notes": null
}
```


In [31]:
# read json
with open("parsed_resp.json", "r") as f:
    parsed_resp = json.load(f)
print(parsed_resp.keys())

dict_keys(['PDEs', 'number_of_state_variables', 'texture_size', 'spatial_step', 'domain_size', 'temporal_step', 'time_horizon', 'boundary_conditions', 'parameter_values', 'notes'])


In [32]:
coding_prompt = coding_agent.coding_prompt
with open('coding_skeleton.frag', 'r') as f:
    coding_skeleton = f.read()

user_prompt = coding_prompt.format(
    PDEs=parsed_resp["PDEs"],
    coding_skeleton = coding_skeleton
)

In [33]:
response = chat(
    model='qwen2.5-coder:3b',
    messages=[
        #{'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ],
)
print(response.message.content)


```glsl

void main() {
    vec2  size  = vec2(textureSize(inTexture, 0)) ;

    vec2 ii = vec2(1.,0.)/size ;
    vec2 jj = vec2(0.,1.)/size ;

    // read the color of the pixel
    vec4 color = texture( inTexture , cc ) ;

    // Standard Laplacian calculation
    float laplacian = (
        texture(inTexture, cc + ii).r + 
        texture(inTexture, cc - ii).r + 
        texture(inTexture, cc + jj).r + 
        texture(inTexture, cc - jj).r - 
        4.0 * u
    ) / (dx*dx) ;

    float dudt = D*laplacian ;

    // Compute I_{fi}(u, v)
    float I_fi = K * u * v;

    // Compute I_{so}(u)
    float I_so = V_csi * pow(u, 3);

    // Compute I_{si}(u, w)
    float I_si = V_v * pow(w, 2) + V_c * w;

    // Compute the right-hand side of the PDE
    vec4 rhs = vec4(I_fi + I_so + I_si, 0.0, 0.0, 0.0);

    // Compute the term (-I_{fi}(u, v) - I_{so}(u) - I_{si}(u, w)) / C_m
    float rhsValue = (I_fi + I_so + I_si) / C_m;

    // Update u using the PDE equation
    u += dt * dudt - rhsVa

In [34]:
#deleat ```glsl at beginning of response.text if exist
response_text = response.message.content
if response_text.startswith("```glsl"):
    response_text = response_text[len("```glsl"):]

# #deleat ``` at end of response.text if exist
if response_text.endswith("```"):
    response_text = response_text[:-3]

with open("march_shader.frag", "w", encoding="utf-8") as f:
    f.write(response_text)